# CIC-IoT 2023 — Policy Pipeline (notebook 8)

Clean-rewrite of the LLM policy generation pipeline. All logic lives in `lib/policy_pipeline/`; this notebook is a thin driver.

**Switch models** by changing `provider` and `model_id` in the config cell. Same code path for Claude Haiku and Gemini 2.5 Flash.

**Hyperparameter sweeps** should use `iot-llm-hids-pg-archive/experiments/sweep_policy.py` (archived; not part of the published pipeline) instead of editing this notebook by hand.

In [ ]:
import sys, time, os
REPO = os.path.abspath(os.path.join(os.getcwd(), '..'))
if REPO not in sys.path:
    sys.path.insert(0, REPO)

import numpy as np
import pandas as pd
from lib.policy_pipeline import RunConfig, run_pipeline
from lib.policy_pipeline.eval import classification_metrics
from lib.policy_pipeline.voting import predict
from lib.policy_pipeline.io import save_policy

## Config

Edit `provider`/`model_id` here to switch LLMs. Everything else is a tunable; defaults match the plan.

In [ ]:
cfg = RunConfig(
    dataset_key='cic',
    provider='anthropic',                    # 'anthropic' | 'google'
    model_id='claude-haiku-4-5-20251001',    # or 'gemini-2.5-flash' for Gemini
    seed=42,
    k=5,
    max_rounds=4,
    early_stop_patience=2,
    voting_mode='weighted',                  # 'majority' | 'or_gate' | 'weighted'
    selection_metric='macro_f1',             # 'macro_f1' | 'attack_f1' | 'attack_precision_at_recall_0.9'
    temperature=0.1,
)
cfg

## Load & split data (identical to notebook 4 for comparability)

In [ ]:
DATA = os.path.join(os.getcwd(), 'data', 'sample-100000-2.csv')
df = pd.read_csv(DATA)
attack_class_labels = df[df['label'] != 'BenignTraffic']['label']
normal = df[df['label'] == 'BenignTraffic'].drop(columns=['label'])
attack = df[df['label'] != 'BenignTraffic'].drop(columns=['label'])
attack_class_labels.index = attack.index

normal_train = normal.sample(frac=0.8, random_state=42)
normal_test  = normal.drop(normal_train.index)
attack_train = attack.sample(frac=0.8, random_state=42)
attack_test  = attack.drop(attack_train.index)
labels_train = attack_class_labels.loc[attack_train.index]
print(f'Train: normal={len(normal_train)}, attack={len(attack_train)}, classes={labels_train.nunique()}')
print(f'Test:  normal={len(normal_test)},  attack={len(attack_test)}')

## Run pipeline

In [ ]:
t0 = time.time()
result = run_pipeline(cfg, normal_train, attack_train, labels_train, verbose=True)
elapsed = time.time() - t0
print(f'\nWall time: {elapsed:.1f}s; tokens in/out: {result.total_tokens_in}/{result.total_tokens_out}')

## Held-out test evaluation

The test set is touched only here, after round selection has completed on the internal validation slice. Fixes the test-set selection bias in notebook 4 cell 13.

In [ ]:
X_te = pd.concat([normal_test, attack_test], axis=0, ignore_index=True)
y_te = np.concatenate([np.full(len(normal_test), 'normal'), np.full(len(attack_test), 'attack')])
pred_te = predict(result.best_policy, X_te)
test_metrics = classification_metrics(y_te, pred_te)

n_distinct_tags = len({r.phenomenon_tag for r in result.best_policy.rules})
print(f'Best round: {result.best_round_index + 1}')
print(f'Policy ({len(result.best_policy.rules)} rules, {n_distinct_tags} distinct tags):')
for r, w in zip(result.best_policy.rules, result.best_policy.weights):
    print(f'  [{r.phenomenon_tag:9}] (w={w:.3f}) {r.feature} {r.op} {r.value}')
    print(f'             {r.rationale}')
print(f'\nVoting: mode={result.best_policy.voting_mode}, tau={result.best_policy.tau:.4f}')
print(f'\nTest metrics:')
for k, v in test_metrics.items():
    print(f'  {k}: {v:.4f}')

## Save policy JSON

Written to `experiments/policies/cic/<run_id>.json`. Consumed by the canonical eval (`experiments/standardize-results.py` — to be refactored in next session) and by paper-writing scripts.

In [ ]:
path = save_policy(cfg, result)
print(f'Saved: {path}')